In [3]:
import sys
from pathlib import Path


PROJECT_ROOT = Path(r"C:\Users\Sherif\Desktop\ECG analysis")

sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
import importlib
import src.train_autoencoder

importlib.reload(src.train_autoencoder)

import torch
import matplotlib.pyplot as plt

from models.UNet1D import UNet1D
from src.dataset import create_dataloaders
from src.train_autoencoder import train_autoencoder, test_autoencoder

In [6]:
import random
import numpy as np
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # pour reproductibilité (important)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [7]:
train_loader, val_loader, test_loader, label_cols, _ = create_dataloaders()

In [8]:
model = UNet1D(in_channels=1, out_channels=12, embedding_dim=32)

In [9]:
train_autoencoder(
    model,
    train_loader,
    val_loader,
    epochs=10,
    lr=1e-3
)


Epoch 1/10


Evaluating AE: 100%|██████████| 69/69 [00:57<00:00,  1.19it/s]


Train Loss (MSE): 0.4367
Val MSE: 0.4214
Val Corr: 0.7410

Epoch 2/10


Evaluating AE: 100%|██████████| 69/69 [00:38<00:00,  1.78it/s]


Train Loss (MSE): 0.4019
Val MSE: 0.4147
Val Corr: 0.7467

Epoch 3/10


Evaluating AE: 100%|██████████| 69/69 [00:40<00:00,  1.72it/s]


... (11 outputs left out)

Evaluating AE: 100%|██████████| 69/69 [01:03<00:00,  1.08it/s]


Train Loss (MSE): 0.3721
Val MSE: 0.3936
Val Corr: 0.7617

Epoch 10/10


Evaluating AE: 100%|██████████| 69/69 [00:46<00:00,  1.48it/s]

Train Loss (MSE): 0.3690
Val MSE: 0.3933
Val Corr: 0.7626


In [10]:
test_autoencoder(model, test_loader)

KeyboardInterrupt: 

In [ ]:
signals, _ = next(iter(test_loader))
signals = signals[:1]

input_lead = signals[:, 0:1, :]
target = signals

pred, _ = model(input_lead)

pred = pred.detach().cpu()
target = target.cpu()

lead = 0  # à changer

plt.plot(target[0, lead], label="True")
plt.plot(pred[0, lead], label="Pred")
plt.legend()
plt.title(f"Lead {lead}")
plt.show()